# 🐥 Baby Pitch Agency - Research Analysis

This notebook provides a comprehensive analysis of the experiment data. It processes all CSV files found in the `data/` directory, extracts acoustic and behavioral metrics, and visualizes the results.

### Key Measures Computed:
- **Pitch Utterances**: Number of distinct vocal events.
- **Pitch Distribution**: Mean, SD, Min, Max, and Percentiles.
- **Timing**: Duration of vocalizations vs. silence.
- **Behavior**: Number of balloons popped and timing.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import os

# Set visual style for publication-ready plots
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data
We load all files in `data/*.csv` and merge them into a single tidy dataframe.

In [ ]:
def load_tidy_data(directory='data'):
    all_files = glob.glob(os.path.join(directory, "*.csv"))
    df_list = []
    
    for filename in all_files:
        df = pd.read_csv(filename)
        # Store the filename as a column for participant/session tracking
        df['source_file'] = os.path.basename(filename)
        df_list.append(df)
    
    if not df_list:
        print("No CSV files found in 'data/' folder.")
        return pd.DataFrame()
    
    return pd.concat(df_list, ignore_index=True)

df_full = load_tidy_data()
print(f"Loaded {df_full['source_file'].nunique()} participant files.")
df_full.head()

## 2. Quantitative Metrics
Calculating descriptive statistics and behavioral outcomes per participant.

In [ ]:
def extract_experiment_metrics(df):
    metrics_list = []
    
    for filename, group in df.groupby('source_file'):
        # 1. Identify Vocalizations
        # Clean data: only strictly True vocalizations with valid pitch
        vocal_mask = (df['is_vocalizing'] == "True") & (df['pitch_hz'] > 0)
        vocal_only = group[group['is_vocalizing'] == "True"]
        
        # 2. Pitch Stats
        p_stats = vocal_only['pitch_hz'].describe(percentiles=[.25, .5, .75])
        
        # 3. Utterance Counting
        # Count changes from False to True
        group_sorted = group.sort_values('timestamp_ms')
        vocal_bin = (group_sorted['is_vocalizing'] == "True").astype(int)
        num_utterances = (vocal_bin.diff() == 1).sum()
        
        # 4. Timing
        # Calculate durations between timestamps
        group_sorted['delta'] = group_sorted['timestamp_ms'].diff().fillna(0)
        vocal_time = group_sorted[group_sorted['is_vocalizing'] == "True"]['delta'].sum()
        silence_time = group_sorted[group_sorted['is_vocalizing'] == "False"]['delta'].sum()
        
        # 5. Balloons
        pops = group_sorted[group_sorted['ballon_popped_id'] != "False"]
        num_pops = pops['ballon_popped_id'].nunique()
        
        metrics_list.append({
            'Participant': filename,
            'Condition': group['condition'].iloc[0],
            'Pitch_Mean': p_stats['mean'],
            'Pitch_SD': p_stats['std'],
            'Pitch_Median': p_stats['50%'],
            'Utterances': num_utterances,
            'Vocalization_Time_Sec': vocal_time / 1000,
            'Silence_Time_Sec': silence_time / 1000,
            'Balloons_Popped': num_pops
        })
        
    return pd.DataFrame(metrics_list)

summary_stats = extract_experiment_metrics(df_full)
summary_stats

## 3. Visualizations

### A. Pitch-to-Height Correlation
Validating how accurately the pitch mapped to the bird's vertical position.

In [ ]:
vocal_data = df_full[df_full['is_vocalizing'] == "True"].copy()

plt.figure(figsize=(10, 8))
sns.regplot(data=vocal_data, x='pitch_hz', y='bird_y', 
            scatter_kws={'alpha':0.1, 'color':'#FF1493'}, 
            line_kws={'color':'#4B0082', 'label':'Correlation Line'})

plt.title("Vocal Control Mapping: Pitch vs. Bird Position", pad=20)
plt.xlabel("Extracted Pitch (Hz)")
plt.ylabel("Bird Y Coordinate (Screen Pixels)")
plt.gca().invert_yaxis()  # Invert because higher Y = lower on screen
plt.legend()
plt.show()

### B. Pitch Contours
Visualizing the trajectory of the vocalizations over time.

In [ ]:
plt.figure(figsize=(14, 6))
sns.lineplot(data=vocal_data, x='timestamp_ms', y='pitch_hz', 
             hue='condition', style='condition', 
             alpha=0.8, linewidth=2)

plt.title("Pitch Contours by Condition")
plt.xlabel("Experiment Time (ms)")
plt.ylabel("Pitch (Hz)")
plt.legend(title="Condition", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### C. Condition Comparisons
Comparing the High vs. Low Contingency groups on key outcome measures.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 1. Mean Pitch Comparison
sns.boxplot(ax=axes[0], data=summary_stats, x='Condition', y='Pitch_Mean', palette='Set2')
sns.swarmplot(ax=axes[0], data=summary_stats, x='Condition', y='Pitch_Mean', color='.25', alpha=0.5)
axes[0].set_title("Mean Pitch Distribution by Condition")
axes[0].set_ylabel("Mean Pitch (Hz)")

# 2. Vocal Activity Comparison
sns.barplot(ax=axes[1], data=summary_stats, x='Condition', y='Vocalization_Time_Sec', palette='Set3')
axes[1].set_title("Total Vocalization Time (Engagement)")
axes[1].set_ylabel("Seconds of Active Vocalization")

plt.tight_layout()
plt.show()